# <img style="float: left; padding-right: 20px; width: 200px" src="https://raw.githubusercontent.com/raxlab/imt2200-data/main/media/logo.jpg">  IMT 2200 - Introducción a Ciencia de Datos
**Pontificia Universidad Católica de Chile**<br>
**Instituto de Ingeniería Matemática y Computacional**<br>
**Semestre 2025-S2**<br>
**Profesor:** Rodrigo A. Carrasco <br>

# <h1><center>Actividad 04: Obteniendo Datos de la Web</center></h1>

Esta actividad busca aplicar conocimientos sobre lectura de datos desde la web en distintos formatos (scrapping y APIs) para la creación de un dataset unificado.

## Instrucciones

Este Notebook contiene las instrucciones a realizar para la actividad. 

<b>Al finalizarla, deben subir el Notebook y los archivos generados en un único archivo .zip, al módulo de la Actividad 04 en Canvas. Entregas posteriores al cierre de la actividad serán evaluadas con nota 1.0.</b>

## Actividad

Para esta actividad, queremos analizar la calidad del aire de las ciudades más pobladas del mundo. Para esto, realice los siguientes pasos:

**1. Extraer datos con web scrapping y API**

Vamos a extraer una lista de las ciudades más pobladas desde Wikipedia, específicamente en el siguiente URL:

`URL_1 = https://en.wikipedia.org/wiki/List_of_largest_cities#List`

Por otra parte, usaremos la [Open-Meteo API](https://open-meteo.com/). Este es un servicio open-source de meteorología que nos permitirá obtener datos como las coordenadas de una ciudad y sus parámetros climáticos, como la calidad del aire.

* 1.1 Utilizando las librerías `requests`y `BeautifulSoup`, obtenga todas las filas y columnas de la tabla de Wikipedia de las ciudades más grandes del mundo y genere un DataFrame a partir de ellas. Su DataFrame debe contener como mínimo las siguientes columnas: ciudad, país y población estimada.

* 1.2 Transforme la columna de población en valores numéricos y sólo deje las 20 mayores ciudades.

**2. Llamada a la API**

* 2.1 Ahora, utilizando `requests`, haga un llamado al siguiente URL de la API de Open-Meteo, reemplazando el valor `CIUDAD` con cada uno de los nombres de las ciudades de su DataFrame:

`URL_2 = https://geocoding-api.open-meteo.com/v1/search?name={CIUDAD}&count=1&language=en&format=json`

Haga una copia de su DataFrame anterior. En esta copia, agregue dos columnas nuevas y guarde los valores obtenidos de latitud y longitud (sin modificar el DataFrame original).

* 2.2 Con los datos de las coordenadas, podemos acceder a información sobre la calidad del aire actual disponible con Open-Meteo. Nuevamente, para todas las ciudades, utilice el URL dado para obtener el índices de calidad del aire (usaremos el europeo) y la cantidad de partículas en suspensión.

`URL_3 = https://air-quality-api.open-meteo.com/v1/air-quality?latitude={LAT}&longitude={LON}&current=european_aqi,pm10,pm2_5`

Guarde los valores obtenidos en nuevas columnas del mismo DataFrame.

* 2.3 En la documentación de Open-Meteo ([aquí](https://open-meteo.com/en/docs/air-quality-api)), podemos ver el significado de los valores del índice European AQI. Utilizando la función `aqi2str()` entregada, genere una nueva columna `Air Quality` (string) a partir de los valores que obtuvo mediante la API.

* 2.4 Revise los valores obtenidos. ¿Tienen sentido? Si hay valores que considere inválidos o "outliers" (extremadamente altos), descártelos del dataset.

**3. Visualización**

Vamos a generar dos visualizaciones a partir de las ciudades con las que hemos trabajado. Para esto, usaremos una nueva librería de visualización llamada `plotly.express`. Plotly permite generar gráficos interactivos, con tooltips donde podemos mostrar información adicional de nuestro DataFrame, lo cual los hace muy convenientes para la exploración de un dataset.

Lea y complete el código entregado con los valores de su DataFrame. Ejecute las celdas y responda:

* Entre las ciudades más pobladas, ¿cuál es la calidad de aire más común?

* ¿Cómo es la relación entre tamaño de población y calidad del aire de las ciudades?

* ¿Hay algún lugar del mundo donde se vea una mayor concentración de grandes ciudades? Si la hay, ¿cómo es la calidad del aire en estas zonas?

## Rúbrica

- Si han hecho todo y sólo hay errores menores: 7.0
- Si sólo llegaron hasta la parte 2.1: 5.0
- Menos que eso: 1.0

### 0. Algunas librerías

Las siguientes son algunas de las librerías que recomendamos usar para esta Actividad. Puede agregar más si lo requiere.

In [105]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import numpy as np

In [94]:
URL_1 = "https://en.wikipedia.org/wiki/List_of_largest_cities#List"
URL_2 = "https://geocoding-api.open-meteo.com/v1/search"
URL_3 = "https://air-quality-api.open-meteo.com/v1/air-quality"

### 1. Extraer datos

#### 1.1:

Utilizando las librerías `requests`y `BeautifulSoup`, obtenga todas las filas y columnas de la tabla de Wikipedia de las ciudades más grandes del mundo y genere un DataFrame a partir de ellas. Su DataFrame debe contener como mínimo las siguientes columnas: ciudad, país y población estimada.

#### 1.1 Respuesta:

In [95]:
# 1.1

# Descargar la página
headers = {"User-Agent": "Mozilla/5.0"}
resp = requests.get(URL_1, headers=headers, timeout=30)
resp.raise_for_status()

soup = BeautifulSoup(resp.text, "html.parser") 

# Buscar la tabla que contenga las columnas City, Country y UN 2018 population estimates
tables = soup.select("table.wikitable.sortable")
#print(tables)
target_table = None

# buscar la tabla específica en la página que contiene las columnas "City", "Country" y "UN 2018 population estimates".
if target_table is None:
    for t in tables:
        first_row = t.find("tr")
        if not first_row:
            continue
        head_text = " ".join(th.get_text(strip=True) for th in first_row.find_all(["th","td"]))
        if all(k in head_text for k in ["City", "Country", "UN 2018 population estimates"]):
            target_table = t
            break


rows = target_table.find_all("tr")
data = []
for r in rows[1:]:
    cells = r.find_all(["td", "th"])
    if len(cells) < 3:
        continue
    # Extraer texto limpio (sin notas [..] ni saltos)
    city = cells[0].get_text(" ", strip=True)
    country = cells[1].get_text(" ", strip=True)
    pop_text = cells[2].get_text(" ", strip=True)

    # Limpiar notas/superscripts y mantener solo dígitos para la población
    pop_num = re.sub(r"[^\d]", "", pop_text)
    if not pop_num:
        continue

    data.append({
        "ciudad": city,
        "pais": country,
        "poblacion_estimada": int(pop_num)
    })

df_ciudades = pd.DataFrame(data).dropna().reset_index(drop=True)
df_ciudades.head(), df_ciudades.shape




(       ciudad        pais  poblacion_estimada
 0  Definition  Population                   2
 1       Tokyo       Japan            37468000
 2       Delhi       India            28514000
 3    Shanghai       China            25582000
 4   São Paulo      Brazil            21650000,
 (82, 3))

#### 1.2:

Transforme la columna de población en valores numéricos y sólo deje las 20 mayores ciudades.

#### 1.2 Respuesta:

In [96]:
# 1.2

col = 'poblacion' if 'poblacion' in df_ciudades.columns else 'poblacion_estimada'

df_top20 = (
    df_ciudades
      .assign(poblacion=pd.to_numeric(df_ciudades[col], errors='coerce'))
      .nlargest(20, 'poblacion')[['ciudad', 'pais', 'poblacion']]
      .reset_index(drop=True)
)

df_top20.head(), df_top20.shape


(        ciudad    pais  poblacion
 0        Tokyo   Japan   37468000
 1        Delhi   India   28514000
 2     Shanghai   China   25582000
 3    São Paulo  Brazil   21650000
 4  Mexico City  Mexico   21581000,
 (20, 3))

### 2. Uso de API

#### 2.1:

Ahora, utilizando `requests`, haga un llamado al siguiente URL de la API de Open-Meteo, reemplazando el valor `CIUDAD` con cada uno de los nombres de las ciudades de su DataFrame:

`URL_2 = https://geocoding-api.open-meteo.com/v1/search?name={CIUDAD}&count=1&language=en&format=json`

Haga una copia de su DataFrame anterior. En esta copia, agregue dos columnas nuevas y guarde los valores obtenidos de latitud y longitud (sin modificar el DataFrame original).

#### 2.1 Respuesta:

In [97]:

# 2.1

# Suponiendo que es df_top20 
df_geo = df_top20.copy()  

def get_lat_lon(city, country):
    try:
        params = {
            "name": f"{city}, {country}",
            "count": 1,
            "language": "en",
            "format": "json"
        }
        r = requests.get(URL_2, params=params, timeout=20)
        r.raise_for_status()
        data = r.json()
        if data.get("results"):
            first = data["results"][0]
            return first.get("latitude"), first.get("longitude")
    except Exception:
        pass
    return None, None

coords = df_geo.apply(lambda s: get_lat_lon(s["ciudad"], s["pais"]), axis=1, result_type=None)
df_geo[["latitud", "longitud"]] = pd.DataFrame(coords.tolist(), index=df_geo.index)

df_geo.head()


,ciudad,pais,poblacion,latitud,longitud
0,Tokyo,Japan,37468000,35.68950,139.69171
1,Delhi,India,28514000,28.65195,77.23149
2,Shanghai,China,25582000,31.22222,121.45806
3,São Paulo,Brazil,21650000,-23.54750,-46.63611
4,Mexico City,Mexico,21581000,19.42847,-99.12766


#### 2.2:

Con los datos de las coordenadas, podemos acceder a información sobre la calidad del aire actual disponible con Open-Meteo. Nuevamente, para todas las ciudades, utilice el URL dado para obtener el índices de calidad del aire (usaremos el europeo) y la cantidad de partículas en suspensión.

`URL_3 = https://air-quality-api.open-meteo.com/v1/air-quality?latitude={LAT}&longitude={LON}&current=european_aqi,pm10,pm2_5`

Guarde los valores obtenidos en nuevas columnas del mismo DataFrame.



#### 2.2 Respuesta:

In [98]:
# 2.2

def get_air_quality(lat, lon):
    if pd.isna(lat) or pd.isna(lon):
        return None, None, None
    try:
        params = {
            "latitude": float(lat),
            "longitude": float(lon),
            "current": "european_aqi,pm10,pm2_5"
        }
        r = requests.get(URL_3, params=params, timeout=20)
        r.raise_for_status()
        cur = r.json().get("current", {})
        return cur.get("european_aqi"), cur.get("pm10"), cur.get("pm2_5")
    except Exception:
        return None, None, None

vals = df_geo.apply(lambda s: get_air_quality(s["latitud"], s["longitud"]), axis=1)
df_geo[["aqi_eu", "pm10", "pm2_5"]] = pd.DataFrame(vals.tolist(), index=df_geo.index)

df_geo.head()

,ciudad,pais,poblacion,latitud,longitud,aqi_eu,pm10,pm2_5
0,Tokyo,Japan,37468000,35.68950,139.69171,45.0,35.3,35.3
1,Delhi,India,28514000,28.65195,77.23149,61.0,46.9,46.2
2,Shanghai,China,25582000,31.22222,121.45806,73.0,95.2,92.0
3,São Paulo,Brazil,21650000,-23.54750,-46.63611,52.0,8.3,8.1
4,Mexico City,Mexico,21581000,19.42847,-99.12766,66.0,39.7,39.3


#### 2.3:

En la documentación de Open-Meteo ([aquí](https://open-meteo.com/en/docs/air-quality-api)), podemos ver el significado de los valores del índice European AQI. Utilizando la función `aqi2str()` entregada, genere una nueva columna `Air Quality` (string) a partir de los valores que obtuvo mediante la API.

#### 2.3 Respuesta:

In [99]:
# ==== CODIGO ENTREGADO - NO MODIFICAR ====
air_quality = {
    "Good": [0, 20],
    "Fair": [20, 40],
    "Moderate": [40, 60],
    "Poor": [60, 80],
    "Very Poor": [80, 100],
    "Extremely Poor": [100, float('inf')]
}

def aqi2str(aqi):
    for key, (low, high) in air_quality.items():
        if low <= aqi < high:
            return key
    return "Unknown"



In [100]:
# 2.3
# Crear columna categórica de calidad del aire a partir de aqi_eu
df_geo["Air Quality"] = df_geo["aqi_eu"].apply(
    lambda v: aqi2str(float(v)) if pd.notna(v) else "Unknown"
)

df_geo.head()

,ciudad,pais,poblacion,latitud,longitud,aqi_eu,pm10,pm2_5,Air Quality
0,Tokyo,Japan,37468000,35.68950,139.69171,45.0,35.3,35.3,Moderate
1,Delhi,India,28514000,28.65195,77.23149,61.0,46.9,46.2,Poor
2,Shanghai,China,25582000,31.22222,121.45806,73.0,95.2,92.0,Poor
3,São Paulo,Brazil,21650000,-23.54750,-46.63611,52.0,8.3,8.1,Moderate
4,Mexico City,Mexico,21581000,19.42847,-99.12766,66.0,39.7,39.3,Poor


#### 2.4: 

Revise los valores obtenidos. ¿Tienen sentido? Si hay valores que considere inválidos o "outliers" (extremadamente altos), descártelos del dataset.

#### 2.4 Respuesta:

In [107]:
# Limpieza: quitar negativos y outliers por IQR

df_geo_clean = df_geo.copy()
for c in cols:
    # quitar negativos
    df_geo_clean.loc[df_geo_clean[c] < 0, c] = np.nan
    # límites IQR (robustos)
    s = df_geo_clean[c].dropna()
    if s.empty:
        continues
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lo = max(0, q1 - 1.5 * iqr)
    hi = q3 + 1.5 * iqr
    df_geo_clean = df_geo_clean[
        df_geo_clean[c].isna() | ((df_geo_clean[c] >= lo) & (df_geo_clean[c] <= hi))
    ]

df_geo_clean = df_geo_clean.reset_index(drop=True)

# Mostrar los registros descartados (si hubo)
descartados = df_geo.merge(df_geo_clean.assign(_keep=1), how="left")
descartados = descartados[descartados["_keep"].isna()][["ciudad","pais"] + cols]
descartados

,ciudad,pais,aqi_eu,pm10,pm2_5
1,Delhi,India,61.0,46.9,46.2
2,Shanghai,China,73.0,95.2,92.0
7,Beijing,China,130.0,155.1,151.3


### 3. Visualizar datos

* ¿Cómo son los valores de calidad de aire para las ciudades más pobladas? ¿Cuál es lo más común?

* ¿Cómo es la relación entre tamaño de población y calidad del aire de una ciudad?

* ¿Hay algún lugar del mundo donde se vea una mayor concentración de grandes ciudades? Si la hay, ¿cómo es la calidad del aire?

In [101]:
# Figura 1: Barplot de calidad del aire
import plotly.express as px

new_df = df_geo

by_quality = new_df.groupby('Air Quality').size().reset_index(name='Count')

fig = px.bar(by_quality,
            x='Air Quality',
            y='Count',
            title="Calidad del aire de 80 ciudades más pobladas del mundo",
            labels={
                "Count": "Cantidad de ciudades",
                "Air Quality": "Calidad del aire"
            },
            color='Air Quality')

fig.update_layout(
    height=400,
    width=900,
)
fig.update_xaxes(categoryorder='array',
                 categoryarray= ["Good", "Fair", "Moderate", "Poor", "Very Poor", "Extremely Poor"]
)
fig.show()

#### Respuesta:

In [102]:
# Figura 2: Scattermap entre población y calidad del aire


fig = px.scatter(df_filtered,
                 x="Population",
                 y="eu_aqi",
                 title="Relación entre población y calidad del aire",
                 labels={
                     "Population": "Población",
                     "eu_aqi": "Calidad del aire (EU AQI)"
                    },
                    hover_data={
                        "City": True,
                        "Country": True,
                        "Population": True,
                        "eu_aqi": True,
                        "Air Quality": True
                        }
                )

fig.update_layout(
    height=500,
    width=900,
)

#### Respuesta:

In [103]:
# Figura 3: Mapa mundial de ciudades más pobladas

fig = px.scatter_geo(data_frame=df_filtered, # Su dataframe
                    lat='lat', # Columna de latitud
                    lon='lon', # Columna de longitud
                    color='eu_aqi', # Columna que representa el color de los puntos
                    hover_name='City', # Columna para el titulo del tooltip
                    projection="natural earth",
                    color_continuous_scale=px.colors.sequential.Inferno_r,
                    title="Calidad del aire de ciudades más pobladas", # Titulo del grafico
                    hover_data={
                        # Qué columnas mostrar en el tooltip
                        "Country": True,
                        "eu_aqi": True,
                        "Air Quality": True
                        # Puede agregar otras...
                    },
                )

fig.update_layout(
    margin={"r":0,"t":50,"l":0,"b":0}, # Márgenes del gráfico
    height=600, # Altura del gráfico
    width=800, # Ancho del gráfico
)
fig.update_traces(
    marker=dict(size=10), # Tamaño de los puntos
)
fig.show()

ValueError: Value of 'lat' is not the name of a column in 'data_frame'. Expected one of ['City', 'Country', 'Population', 'latitud', 'longitud', 'eu_aqi', 'pm10', 'pm2_5', 'Air Quality'] but received: lat

#### Respuesta: